## Augmented LLM


![](https://docs.dapr.io/images/dapr-agents/agents-augmented-llm.png)

Quelle: dapr.io

---

Das Augmented-LLM-Muster ist der grundlegende Baustein für jedes agentenbasierte System. Es erweitert ein Sprachmodell um externe Funktionen wie Speicher und Werkzeuge und bietet so eine einfache, aber leistungsstarke Grundlage für KI-gestützte Anwendungen.

Dieses Muster eignet sich ideal für Szenarien, in denen Sie ein LLM mit erweiterten Funktionen benötigen, aber keine komplexe Orchestrierung oder autonome Entscheidungsfindung erfordern. Das erweiterte LLM kann auf externe Tools zugreifen, den Gesprächsverlauf speichern und konsistente Antworten über verschiedene Interaktionen hinweg liefern.

Anwendungsfälle:
* Persönliche Assistenten, die sich die Benutzerpräferenzen merken
* Kundendienstmitarbeiter, die auf Produktinformationen zugreifen
* Recherchewerkzeuge, die Informationen abrufen und analysieren

Implementierung mit Dapr-Agenten:

In [ ]:
from dataclasses import dataclass, asdict
from dapr_agents import DurableAgent, tool
from dapr_agents.workflow.runners import AgentRunner
from dapr_agents.llm import DaprChatClient

@dataclass
class FlightOption:
    airline: str
    price: float

@tool
def search_flights(destination: str) -> list[dict]:
    """Search for flights to the specified destination."""
    return [
        asdict(FlightOption(airline="SkyHighAir", price=450.00)),
        asdict(FlightOption(airline="GlobalWings", price=375.50)),
    ]

travel_planner = DurableAgent(
    name="TravelBuddy",
    role="Travel Planner Assistant",
    instructions=[
        "Help users find flights.",
        "Use the search_flights tool when the user asks for flight options."
    ],
    tools=[search_flights],
    llm=DaprChatClient(component_name="openai"),
)

runner = AgentRunner()

try:
    result = await runner.run(
        travel_planner,
        payload={"task": "Find flights to Paris"}
    )
    print(result)
finally:
    runner.shutdown()